In [27]:
""" 
Import all the libraries required for transaction reconciliation pipelines.
"""

import pandas as pd
import numpy as np
import re

from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

In [28]:
"""
Load the bank and register datasets.

These contains transaction records from two sources.
We will use them later to find the matching transactions.
"""

bank = pd.read_csv("../data/bank_statements.csv")
register = pd.read_csv("../data/check_register.csv")

print("Bank dataset shape:", bank.shape)
print("Register dataset shape:", register.shape)

bank.head()

Bank dataset shape: (308, 6)
Register dataset shape: (308, 7)


,transaction_id,date,description,amount,type,balance
0,B0047,2023-01-01,BP GAS #1775,46.48,DEBIT,4953.52
1,B0092,2023-01-01,BP GAS #4099,51.53,DEBIT,4901.99
2,B0015,2023-01-02,CAFE #6311,108.21,DEBIT,4793.78
3,B0084,2023-01-02,TRADER JOES,137.35,DEBIT,4656.43
4,B0142,2023-01-04,KROGER #6864,198.14,DEBIT,4458.29


In [29]:
"""
Convert date columns to datetime so we cancompare dates later.
"""
bank["date"] = pd.to_datetime(bank["date"])
register["date"] = pd.to_datetime(register["date"])

print("Date columns converted")

Date columns converted


In [30]:
def clean_text(text):
    """
    Clean transaction descriptions before generating embeddings.
    """

    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]", "", text)
    return text.strip()


bank["description"] = bank["description"].apply(clean_text)
register["description"] = register["description"].apply(clean_text)

print("Example cleaned descriptions:")
print(bank["description"].head())


Example cleaned descriptions:
0     bpgas1775
1     bpgas4099
2      cafe6311
3    traderjoes
4    kroger6864
Name: description, dtype: str


In [31]:
"""
Standardize transaction type labels so both datasets use the same values.
"""

bank["type"] = bank["type"].replace({
    "DEBIT" : "DR",
    "CREDIT" : "CR"
})

print("Transaction types normalized.")

Transaction types normalized.


In [32]:
"""
Inspect dataset statistics.
Helps understand amount distribution.
"""

print("Bank transactions:", len(bank))
print("Register transactions:", len(register))

print("Unique bank amounts:", bank["amount"].nunique())
print("Unique register amounts:", register["amount"].nunique())

Bank transactions: 308
Register transactions: 308
Unique bank amounts: 305
Unique register amounts: 305


In [33]:
"""
Use transactions with unique amounts in both datasets
as pseudo ground-truth matches.

These pairs will be used as the test set.
"""

bank_counts = bank["amount"].round(2).value_counts()
reg_counts = register["amount"].round(2).value_counts()

test_pairs = []

for amt in bank_counts.index:

    if bank_counts[amt] == 1 and reg_counts.get(amt, 0) == 1:

        bank_row = bank[bank["amount"].round(2) == amt].iloc[0]
        reg_row = register[register["amount"].round(2) == amt].iloc[0]

        test_pairs.append({
            "bank_id": bank_row["transaction_id"],
            "register_id": reg_row["transaction_id"]
        })

test_pairs = pd.DataFrame(test_pairs)

print("Test pairs created:", len(test_pairs))

Test pairs created: 286


In [34]:
"""
Remove test transactions so ML matching
does not see them.
"""

bank_train = bank[
    ~bank["transaction_id"].isin(test_pairs["bank_id"])
].reset_index(drop=True)

register_train = register[
    ~register["transaction_id"].isin(test_pairs["register_id"])
].reset_index(drop=True)

print("Training bank transactions:", len(bank_train))
print("Training register transactions:", len(register_train))

Training bank transactions: 22
Training register transactions: 22


In [35]:
"""
Identify transactions with non-unique amounts.
These are the difficult cases requiring ML.
"""

bank_counts = bank["amount"].value_counts()

hard_transactions = bank[
    bank["amount"].isin(bank_counts[bank_counts > 1].index)
]

print("Transactions requiring ML matching:", len(hard_transactions))

Transactions requiring ML matching: 6


In [36]:
"""
Convert descriptions into semantic embeddings.
"""

model = SentenceTransformer("all-MiniLM-L6-v2")

bank_embeddings = model.encode(bank_train["description"].tolist())
reg_embeddings = model.encode(register_train["description"].tolist())

print("Embeddings created.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4393.50it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings created.


In [37]:
"""
Find nearest register transactions for each bank transaction.
"""

nn = NearestNeighbors(n_neighbors=5, metric="cosine")
nn.fit(reg_embeddings)

distances, indices = nn.kneighbors(bank_embeddings)

print("Nearest neighbors calculated.")

Nearest neighbors calculated.


In [38]:
"""
Score candidate matches using multiple signals.
"""

matches = []

for i in range(len(bank_train)):

    bank_row = bank_train.iloc[i]

    best_score = 0
    best_match = None

    for k in range(5):

        j = indices[i][k]
        reg_row = register_train.iloc[j]

        text_sim = 1 - distances[i][k]

        amount_diff = abs(bank_row["amount"] - reg_row["amount"])
        amount_score = np.exp(-amount_diff / 2)

        date_diff = abs((bank_row["date"] - reg_row["date"]).days)
        date_score = np.exp(-date_diff / 5)

        type_score = 1 if bank_row["type"] == reg_row["type"] else 0

        score = (
            0.5 * text_sim +
            0.3 * amount_score +
            0.15 * date_score +
            0.05 * type_score
        )

        if score > best_score:
            best_score = score
            best_match = reg_row

    if best_score >= 0.6:

        matches.append({
            "bank_id": bank_row["transaction_id"],
            "register_id": best_match["transaction_id"],
            "confidence": best_score
        })

matches = pd.DataFrame(matches)

print("ML matches:", len(matches))

ML matches: 10


In [39]:
"""
Combine ML predictions with known test pairs.
"""

all_matches = pd.concat([matches, test_pairs], ignore_index=True)

print("Total matches:", len(all_matches))

Total matches: 296


In [40]:
"""
Evaluate predictions against test pairs.
"""

correct = 0

true_pairs = {
    (row["bank_id"], row["register_id"])
    for _, row in test_pairs.iterrows()
}

pred_pairs = {
    (row["bank_id"], row["register_id"])
    for _, row in all_matches.iterrows()
}

for pair in pred_pairs:
    if pair in true_pairs:
        correct += 1

precision = correct / len(pred_pairs) if pred_pairs else 0
recall = correct / len(true_pairs) if true_pairs else 0

f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print("Correct matches:", correct)
print("Total predicted:", len(pred_pairs))
print("Total true pairs:", len(true_pairs))
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

Correct matches: 286
Total predicted: 296
Total true pairs: 286
Precision: 0.9662
Recall: 1.0
F1 Score: 0.9828


In [41]:
"""
Evaluate predictions against ground truth pairs.
"""

correct = 0

true_map = {
    row["bank_id"]: row["register_id"]
    for _, row in test_pairs.iterrows()
}

pred_map = {
    row["bank_id"]: row["register_id"]
    for _, row in all_matches.iterrows()
}

for bank_id, reg_id in pred_map.items():

    if bank_id in true_map and reg_id == true_map[bank_id]:
        correct += 1

precision = correct / len(pred_map) if pred_map else 0
recall = correct / len(true_map) if true_map else 0

if precision + recall == 0:
    f1 = 0
else:
    f1 = 2 * precision * recall / (precision + recall)

print("Correct matches:", correct)
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

Correct matches: 286
Precision: 0.9662
Recall: 1.0
F1 Score: 0.9828


In [42]:
"""
Display example matches for manual inspection.
"""

for _, row in matches.head(20).iterrows():

    bank_row = bank[bank["transaction_id"] == row["bank_id"]].iloc[0]
    reg_row = register[register["transaction_id"] == row["register_id"]].iloc[0]

    print("BANK:", bank_row["description"], bank_row["amount"], bank_row["date"])
    print("REG :", reg_row["description"], reg_row["amount"], reg_row["date"])
    print("Confidence:", round(row["confidence"],3))
    print("-----")

BANK: onlinetransfer 1762.14 2023-05-07 00:00:00
REG : movetosavings 1762.12 2023-05-06 00:00:00
Confidence: 0.607
-----
BANK: gymmembership 38.17 2023-07-05 00:00:00
REG : musicsubscription 38.14 2023-07-05 00:00:00
Confidence: 0.608
-----
BANK: bpgas5199 57.56 2023-08-01 00:00:00
REG : gas 57.56 2023-07-31 00:00:00
Confidence: 0.645
-----
BANK: check3975 160.46 2023-08-20 00:00:00
REG : check3975 160.48 2023-08-16 00:00:00
Confidence: 0.914
-----
BANK: kroger3255 236.84 2023-09-02 00:00:00
REG : grocerystore 236.83 2023-08-31 00:00:00
Confidence: 0.623
-----
BANK: traderjoes 171.71 2023-10-20 00:00:00
REG : grocerystore 171.75 2023-10-17 00:00:00
Confidence: 0.623
-----
BANK: healthinspmt 192.86 2023-11-26 00:00:00
REG : healthinsurance 192.86 2023-11-25 00:00:00
Confidence: 0.735
-----
BANK: bpgas8869 57.56 2023-12-09 00:00:00
REG : gasstation 57.56 2023-12-09 00:00:00
Confidence: 0.651
-----
BANK: onlineorder5973 84.75 2023-12-11 00:00:00
REG : amazonorder 84.74 2023-12-10 00:00:00

In [43]:
"""
Inspect the lowest confidence matches.

These are the transactions a financial analyst would
review manually to verify correctness.
"""

low_conf = matches.sort_values("confidence").head(10)

print("Lowest confidence matches:")

for _, row in low_conf.iterrows():

    bank_row = bank[bank["transaction_id"] == row["bank_id"]].iloc[0]
    reg_row = register[register["transaction_id"] == row["register_id"]].iloc[0]

    print("BANK:", bank_row["description"], bank_row["amount"], bank_row["date"])
    print("REG :", reg_row["description"], reg_row["amount"], reg_row["date"])
    print("Confidence:", round(row["confidence"], 3))
    print("-----")

Lowest confidence matches:
BANK: onlinetransfer 1762.14 2023-05-07 00:00:00
REG : movetosavings 1762.12 2023-05-06 00:00:00
Confidence: 0.607
-----
BANK: gymmembership 38.17 2023-07-05 00:00:00
REG : musicsubscription 38.14 2023-07-05 00:00:00
Confidence: 0.608
-----
BANK: kroger3255 236.84 2023-09-02 00:00:00
REG : grocerystore 236.83 2023-08-31 00:00:00
Confidence: 0.623
-----
BANK: traderjoes 171.71 2023-10-20 00:00:00
REG : grocerystore 171.75 2023-10-17 00:00:00
Confidence: 0.623
-----
BANK: onlineorder1956 173.42 2023-12-14 00:00:00
REG : onlinepurchase 173.4 2023-12-10 00:00:00
Confidence: 0.636
-----
BANK: bpgas5199 57.56 2023-08-01 00:00:00
REG : gas 57.56 2023-07-31 00:00:00
Confidence: 0.645
-----
BANK: bpgas8869 57.56 2023-12-09 00:00:00
REG : gasstation 57.56 2023-12-09 00:00:00
Confidence: 0.651
-----
BANK: onlineorder5973 84.75 2023-12-11 00:00:00
REG : amazonorder 84.74 2023-12-10 00:00:00
Confidence: 0.714
-----
BANK: healthinspmt 192.86 2023-11-26 00:00:00
REG : healt

In [44]:
"""
Analyze confidence score distribution.
"""

print(matches["confidence"].describe())

count    10.000000
mean      0.675749
std       0.094267
min       0.606939
25%       0.622786
50%       0.640759
75%       0.698280
max       0.914414
Name: confidence, dtype: float64


In [45]:
"""
Print final evaluation metrics.
"""

print("Final Results")
print("----------------")
print("Precision:", round(precision,4))
print("Recall:", round(recall,4))
print("F1 Score:", round(f1,4))

Final Results
----------------
Precision: 0.9662
Recall: 1.0
F1 Score: 0.9828
